# EasyRAG Retrieval Failure Cases And Debugging

## What you will do

- trigger several retrieval failures on purpose
- inspect `candidate_counts`, `filters_applied`, `stage_timings_ms`, and `fallback_used`
- compare parameter changes that recover or worsen retrieval quality

## Why this matters

Retrieval debugging gets easier when you can localize the failure: weak query coverage, overly strict filters, backend fallback, or just a low-quality corpus.


## Source code anchors

- `easyrag.rag.retrieval.pipeline.execute_query`
- `easyrag.rag.types.QueryParam`
- `tests.test_rag_generation_eval.test_query_filters_and_observability_metadata_are_exposed`


In [ ]:
# ruff: noqa: E402,F401,F811
import sys
from pathlib import Path

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "easyrag").exists():
        REPO_ROOT = candidate.resolve()
        if str(REPO_ROOT) not in sys.path:
            sys.path.insert(0, str(REPO_ROOT))
        break
else:
    raise RuntimeError("Could not locate the EasyRAG repository root.")

import json
import os
import tempfile
import time
from pathlib import Path
from pprint import pprint
from unittest import mock

from easyrag.rag import AnswerParam, EasyRAG, EvalCase, QueryParam
from easyrag.support.async_utils import run_sync
from notebooks._utils import (
    managed_demo_rag,
    print_json as _print_json,
    production_backends_ready,
    provider_ready as can_use_openai_compatible_models,
    skip_message,
    stub_embedding as _stub_embedding,
    stub_query_model as _stub_query_model,
    stub_reranker as _stub_reranker,
)


## Deterministic path

We will use the same corpus and trigger three different failure modes: metadata over-filtering, score thresholding, and backend fallback.


In [ ]:
debug_tmp = tempfile.TemporaryDirectory()
debug_rag = EasyRAG(
    working_dir=debug_tmp.name,
    workspace="retrieval-debug-demo",
    embedding_func=_stub_embedding,
    query_model_func=_stub_query_model,
)
run_sync(debug_rag.initialize_storages())
run_sync(
    debug_rag.ainsert(
        [
            "# Architecture\nretrieval retrieval retrieval keeps architecture grounded.\n",
            "# Guide\nretrieval keeps guides readable.\n",
        ],
        ids=["doc::architecture", "doc::guide"],
        file_paths=["docs/architecture.md", "docs/guide.md"],
    )
)

healthy = run_sync(debug_rag.aquery("retrieval", QueryParam(mode="naive", rewrite_enabled=False, mqe_enabled=False)))
over_filtered = run_sync(debug_rag.aquery("retrieval", QueryParam(mode="naive", rewrite_enabled=False, mqe_enabled=False, metadata_filters={"doc_id": "doc::missing"})))
thresholded = run_sync(debug_rag.aquery("retrieval retrieval", QueryParam(mode="naive", rewrite_enabled=False, mqe_enabled=False, min_score=130.0)))

_print_json({
    "healthy": {"metadata": healthy.metadata, "citations": healthy.citations},
    "over_filtered": {"metadata": over_filtered.metadata, "citations": over_filtered.citations},
    "thresholded": {"metadata": thresholded.metadata, "citations": thresholded.citations},
})


## Output inspection

The next cell swaps in a failing embedding function so you can see the retrieval diagnostics when the vector layer falls back to token overlap.


In [ ]:
def failing_embedding(texts):
    raise RuntimeError("embedding backend unavailable")

fallback_tmp = tempfile.TemporaryDirectory()
fallback_rag = EasyRAG(
    working_dir=fallback_tmp.name,
    workspace="retrieval-debug-fallback-demo",
    embedding_func=failing_embedding,
    query_model_func=_stub_query_model,
)
run_sync(fallback_rag.initialize_storages())
run_sync(fallback_rag.ainsert("# Retrieval\nretrieval retrieval keeps the fallback demo visible.\n", ids=["doc::retrieval"], file_paths=["docs/retrieval.md"]))
fallback_result = run_sync(fallback_rag.aquery("retrieval", QueryParam(mode="naive", rewrite_enabled=False, mqe_enabled=False)))
_print_json({"metadata": fallback_result.metadata, "citations": fallback_result.citations})


## Provider-backed path

The optional path keeps the same debugging habit but runs against the configured provider-backed stack when available.


In [ ]:
if not can_use_openai_compatible_models():
    print("Skipping provider-backed path because OPENAI-compatible config is not set.")
else:
    provider_tmp = tempfile.TemporaryDirectory()
    provider_rag = EasyRAG(working_dir=provider_tmp.name, workspace="provider-retrieval-debug-demo")
    try:
        run_sync(provider_rag.initialize_storages())
        run_sync(provider_rag.ainsert("# Retrieval\nGrounded retrieval stays inspectable.\n", ids=["doc::retrieval"], file_paths=["docs/retrieval.md"]))
        provider_result = run_sync(provider_rag.aquery("retrieval", QueryParam(mode="hybrid", chunk_top_k=2)))
        _print_json(provider_result.metadata)
    except Exception as exc:
        print(f"Provider-backed path was skipped due to runtime error: {exc}")
    finally:
        try:
            run_sync(provider_rag.finalize_storages())
        finally:
            provider_tmp.cleanup()


## What changed and why

Each failure leaves a different trace. Over-filtering reduces `selected_post_filter` and often empties the final citations. High score thresholds can silently cut away useful evidence. Backend fallback changes the `vector_backend` label and sets `fallback_used`. These are different bugs, and the metadata helps you tell them apart.


In [ ]:
run_sync(debug_rag.finalize_storages())
debug_tmp.cleanup()
run_sync(fallback_rag.finalize_storages())
fallback_tmp.cleanup()
print("Cleaned up the retrieval-debug workspaces.")


## Next steps

- Continue with [06_03_retrieval_metrics.ipynb](../06_evaluation/06_03_retrieval_metrics.ipynb) to turn these debugging habits into systematic retrieval evaluation.
- Read [principles/21-evaluation-and-debugging.md](../../docs/principles/21-evaluation-and-debugging.md) for the broader debugging mindset.
- Revisit [04_06_hydration_and_citations.ipynb](04_06_hydration_and_citations.ipynb) if you want to inspect the evidence objects that these failures produce.
